# End-to-end per-dataset attack ablation

This notebook generates per-dataset attacks for MVTec and VisA and evaluates them on AnomalyCLIP. The full matrix is 500/800 steps × 2/255 and 4/255 × the `ce_focal_dice` and `margin_topk` loss formulations, followed by fixed-0.5, image-F1, and clean-pixel-F1 threshold evaluations. Enable a GPU and Internet. **The complete matrix is longer than one Kaggle T4 session; select one setup per saved Kaggle run unless using a longer-running server.** Running both loss formulations doubles the work inside a setup, so a single setup can be split across two sessions with `LOSS_FORMULATIONS`: each formulation writes its own `setups/<setup>/<formulation>/` folder, so two sessions accumulate instead of overwriting each other.

In [ ]:
# Clone the exact experiment code
import os, subprocess, sys
from pathlib import Path
WORKING = Path('/kaggle/working')
REPO_ROOT = WORKING / 'adversarial-robustness'
REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
REPO_REF = 'main'  # Replace with the final commit hash before sharing results.
if not (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', '--force', 'FETCH_HEAD'], check=True)
PIPELINE = REPO_ROOT / 'per_dataset_ablation_pipeline'
print('Commit:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('Pipeline:', PIPELINE)

In [ ]:
# Select and resolve datasets
import torch
DATASETS = 'mvtec,visa'  # Choose: 'mvtec', 'visa', or 'mvtec,visa'
if DATASETS not in {'mvtec', 'visa', 'mvtec,visa'}:
    raise ValueError("DATASETS must be 'mvtec', 'visa', or 'mvtec,visa'")
def find_mvtec():
    for candidate in Path('/kaggle/input').rglob('mvtec_anomaly_detection'):
        if (candidate / 'bottle' / 'test').is_dir():
            return candidate.resolve()
    for test_dir in Path('/kaggle/input').rglob('test'):
        if test_dir.parent.name == 'bottle' and (test_dir.parent.parent / 'carpet' / 'test').is_dir():
            return test_dir.parent.parent.resolve()
    raise FileNotFoundError('Attach MVTec AD.')
MVTEC_ROOT = find_mvtec() if 'mvtec' in DATASETS.split(',') else Path('/kaggle/working/unused_mvtec')
# VisA root contains split_csv and category folders.
visa_candidates = [p for p in Path('/kaggle/input').rglob('VisA_20220922') if (p / 'split_csv').is_dir()]
if not visa_candidates:
    visa_candidates = [p.parent.parent for p in Path('/kaggle/input').rglob('1cls.csv') if p.parent.name == 'split_csv']
if 'visa' in DATASETS.split(',') and not visa_candidates:
    raise FileNotFoundError('Attach the VisA dataset.')
VISA_ROOT = visa_candidates[0].resolve() if visa_candidates else Path('/kaggle/working/unused_visa')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
print('GPU:', torch.cuda.get_device_name(0))
print('Selected datasets:', DATASETS)
if 'mvtec' in DATASETS.split(','): print('MVTec:', MVTEC_ROOT)
if 'visa' in DATASETS.split(','): print('VisA:', VISA_ROOT)

In [ ]:
# Experiment controls
# Use '' on a long-running server. On Kaggle, run one setup per saved session:
# steps500_eps2, steps500_eps4, steps800_eps2, or steps800_eps4
RUN_SETUPS = 'all'
RUN_PHASE = 'all'  # Use 'evaluate' to resume after generation has completed.
# Objective families: 'ce_focal_dice,margin_topk' runs both and doubles the work
# inside every setup. Use one value to split a setup across two sessions; each
# formulation has its own setups/<setup>/<formulation>/ folder, so nothing is
# overwritten and the final audit collects both.
LOSS_FORMULATIONS = 'ce_focal_dice,margin_topk'
# K for TopK(H) per attack direction, as a fraction of the patch tokens
# (518/14 = 37 per side, so 1369 tokens with the ViT-L/14 surrogate).
TOPK_FRACTION_NORMAL_TO_ABNORMAL = 0.20  # Plant a fake defect: ~274 tokens.
TOPK_FRACTION_ABNORMAL_TO_NORMAL = 0.40  # Suppress a real defect: ~548 tokens.
SMOKE_TEST = False  # True validates plumbing with 2 steps; never report smoke results.
GENERATION_BATCH_SIZE = 2  # T4-safe; increase only with more GPU memory.
EVALUATION_BATCH_SIZE = 1  # Separate AnomalyCLIP evaluation batch size.
OUTPUT_ROOT = WORKING / 'per_dataset_ablation_outputs'
valid_formulations = {'ce_focal_dice', 'margin_topk'}
selected_formulations = [name.strip() for name in LOSS_FORMULATIONS.split(',') if name.strip()]
if not selected_formulations or not set(selected_formulations) <= valid_formulations:
    raise ValueError(f'LOSS_FORMULATIONS must be a subset of {sorted(valid_formulations)}')
for label, fraction in (
    ('TOPK_FRACTION_NORMAL_TO_ABNORMAL', TOPK_FRACTION_NORMAL_TO_ABNORMAL),
    ('TOPK_FRACTION_ABNORMAL_TO_NORMAL', TOPK_FRACTION_ABNORMAL_TO_NORMAL),
):
    if not 0.0 < fraction <= 1.0:
        raise ValueError(f'{label} must be in (0, 1]')
env = os.environ.copy()
env.update({
    'MVTEC_ROOT': str(MVTEC_ROOT),
    'VISA_ROOT': str(VISA_ROOT),
    'PIPELINE_OUTPUT': str(OUTPUT_ROOT),
    'DATASETS': DATASETS,
    'RUN_SETUPS': RUN_SETUPS,
    'RUN_PHASE': RUN_PHASE,
    'LOSS_FORMULATIONS': ','.join(selected_formulations),
    'MARGIN_TOPK_FRACTION_NORMAL_TO_ABNORMAL': str(TOPK_FRACTION_NORMAL_TO_ABNORMAL),
    'MARGIN_TOPK_FRACTION_ABNORMAL_TO_NORMAL': str(TOPK_FRACTION_ABNORMAL_TO_NORMAL),
    'SMOKE_TEST': str(SMOKE_TEST).lower(),
    'PER_DATASET_BATCH_SIZE': str(GENERATION_BATCH_SIZE),
    'EVALUATION_BATCH_SIZE': str(EVALUATION_BATCH_SIZE),
    'PYTHON_BIN': sys.executable,
})
subprocess.run(['bash', str(PIPELINE / 'train.sh')], cwd=PIPELINE, env=env, check=True)
RESULTS_ROOT = OUTPUT_ROOT / ('datasets_' + DATASETS.replace(',', '_'))
print('Output:', RESULTS_ROOT)

In [ ]:
# Inspect and package completed results
import pandas as pd, shutil
summary_path = RESULTS_ROOT / 'ablation_high_level_summary.csv'
if summary_path.is_file():
    summary = pd.read_csv(summary_path)
    display(summary)
archive = shutil.make_archive(str(RESULTS_ROOT), 'zip', root_dir=RESULTS_ROOT.parent, base_dir=RESULTS_ROOT.name)
print('Downloadable archive:', archive)